In [4]:
import pandas as pd
import sqlite3
from pathlib import Path
import math
from datetime import datetime, timedelta

CLEANED_PATH = Path(r"D:\TechTrech\Tasks\Final Project\ai4i+2020+predictive+maintenance+dataset\ai4i2020_cleaned.csv")
SQL_PATH = Path(r"D:\TechTrech\Tasks\Final Project\sql\schema.sql")
DB_PATH = Path(r"D:\TechTrech\Tasks\Final Project\industrial_maintenance.db")

print("Paths OK")

Paths OK


In [5]:
df = pd.read_csv(r"D:\TechTrech\Tasks\Final Project\ai4i+2020+predictive+maintenance+dataset\data\ai4i2020_cleaned.csv")
print(df.shape)
df.head()

(10000, 17)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,Type_H,Type_L,Type_M
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,0.0,0.0,1.0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,0.0,1.0,0.0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,0.0,1.0,0.0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,0.0,1.0,0.0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,0.0,1.0,0.0


In [6]:
# 1. machines table
machines_df = df[['UDI','Product ID' , 'Type']].copy()
machines_df = machines_df.rename(columns={'UDI':'machine_id', 'Product ID':'product_id'})
print(f"machines: {machines_df.shape}")
machines_df.head()

# 2. sensor_readings table
def calc_power(torque, rpm):
    return torque * rpm * 2 * math.pi / 60

def gen_timestamp(uid):
    
    start = datetime(2020,1,1)
    return start + timedelta(minutes=int(uid)*5)

sensor_df = pd.DataFrame()
sensor_df['reading_id'] = df['UDI']
sensor_df['machine_id'] = df['UDI']
sensor_df['timestamp'] = df['UDI'].apply(gen_timestamp)
sensor_df['air_temperature_k'] = df['Air temperature [K]']
sensor_df['process_temperature_k'] = df['Process temperature [K]']
sensor_df['rotational_speed_rpm'] = df['Rotational speed [rpm]']
sensor_df['torque_nm'] = df['Torque [Nm]']
sensor_df['tool_wear_min'] = df['Tool wear [min]']
sensor_df['power_w'] = df.apply(lambda r: calc_power(r['Torque [Nm]'], r['Rotational speed [rpm]']), axis=1)
sensor_df['temp_diff_k'] = df['Process temperature [K]'] - df['Air temperature [K]']

print(f"sensor_readings: {sensor_df.shape}")
sensor_df.head()

# 3. failure_events table (only failures)
failure_rows = []
for _, row in df[df['Machine failure']==1].iterrows():
    udi = row['UDI']
    modes = []
    if row['TWF']==1: modes.append('TWF')
    if row['HDF']==1: modes.append('HDF')
    if row['PWF']==1: modes.append('PWF')
    if row['OSF']==1: modes.append('OSF')
    if row['RNF']==1: modes.append('RNF')
    if not modes: modes = ['MULTI']
    
    for mode in modes:
        failure_rows.append({
            'machine_id': udi,
            'reading_id': udi,
            'timestamp': gen_timestamp(udi),
            'machine_failure': 1,
            'failure_type': mode,
            'twf': int(row['TWF']),
            'hdf': int(row['HDF']),
            'pwf': int(row['PWF']),
            'osf': int(row['OSF']),
            'rnf': int(row['RNF']),
            'description': f"{mode} failure"
        })

import pandas as pd
failure_df = pd.DataFrame(failure_rows)
print(f"failure_events: {failure_df.shape} (from 339 failures)")
failure_df.head()

machines: (10000, 3)
sensor_readings: (10000, 10)
failure_events: (364, 11) (from 339 failures)


,machine_id,reading_id,timestamp,machine_failure,failure_type,twf,hdf,pwf,osf,rnf,description
0,51,51,2020-01-01 04:15:00,1,PWF,0,0,1,0,0,PWF failure
1,70,70,2020-01-01 05:50:00,1,PWF,0,0,1,1,0,PWF failure
2,70,70,2020-01-01 05:50:00,1,OSF,0,0,1,1,0,OSF failure
3,78,78,2020-01-01 06:30:00,1,TWF,1,0,0,0,0,TWF failure
4,161,161,2020-01-01 13:25:00,1,OSF,0,0,0,1,0,OSF failure


In [7]:
# %%
print("Nulls in machines_df:")
print(machines_df.isnull().sum())
print("\nUnique types:")
print(machines_df['Type'].unique()[:20])
print("\nRows where Type is null or empty:")
print(machines_df[machines_df['Type'].isnull() | (machines_df['Type'].astype(str).str.strip() == '')])

Nulls in machines_df:
machine_id    0
product_id    0
Type          0
dtype: int64

Unique types:
<StringArray>
['M', 'L', 'H']
Length: 3, dtype: str

Rows where Type is null or empty:
Empty DataFrame
Columns: [machine_id, product_id, Type]
Index: []


In [8]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Read and execute schema.sql
with open(r"D:\TechTrech\Tasks\Final Project\ai4i+2020+predictive+maintenance+dataset\sql\schema.sql", 'r') as f:
    schema_sql = f.read()

cursor.executescript(schema_sql)
print("Schema created!")

# Load data into tables
machines_df.to_sql('machines', conn, if_exists='append', index=False)
sensor_df.to_sql('sensor_readings', conn, if_exists='append', index=False)
failure_df.to_sql('failure_events', conn, if_exists='append', index=False)

conn.commit()

# Check counts
for table in ['machines','sensor_readings','failure_events']:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} rows")

conn.close()
print(f"\nDB created at: {DB_PATH}")

Schema created!
machines: 10000 rows
sensor_readings: 10000 rows
failure_events: 364 rows

DB created at: D:\TechTrech\Tasks\Final Project\industrial_maintenance.db


In [9]:
import sqlite3
conn = sqlite3.connect(DB_PATH)

# Test JOIN query
query = """
SELECT m.machine_id, m.product_id, m.type, s.torque_nm, f.failure_type
FROM machines m
JOIN sensor_readings s ON m.machine_id = s.machine_id
LEFT JOIN failure_events f ON s.reading_id = f.reading_id
WHERE f.machine_failure = 1
LIMIT 5
"""

pd.read_sql_query(query, conn)


,machine_id,product_id,type,torque_nm,failure_type
0,51,L47230,L,4.6,PWF
1,70,L47249,L,65.7,PWF
2,70,L47249,L,65.7,OSF
3,78,L47257,L,41.3,TWF
4,161,L47340,L,60.7,OSF
